# Assignment 3: LLMs and Machine Learning

---

## Submission Instructions

Submit only a link to the folder for Assignment 3 in your personal GitHub repository. Within the repository, you should have a Jupyter notebook file titled e.g. `assignment3.ipynb` or something similar, placed inside the `assignments/assignment3/` folder.

Make sure the repository is public.

**Submissions must be made using a GitHub repository. Submissions that do not follow this instruction will receive 0 points.**

**Late submissions are not accepted as the peer review system does not allow adding submissions past the deadline. Submit your work early to not miss the deadline!**

## Code Quality

Write your code so that it is pleasant to read and easy to understand. This includes:

- Use descriptive variable and function names.
- Add brief comments where the logic is not immediately obvious.
- Keep your notebook organized with clear separation between tasks.
- Print out your answers so that the peer reviewer can see the results. Use the `df.head()` when asked to print the top  5 lines. To print a better looking DataFrame, consider also using `display()` instead of `print()`.
- Divide the code into logical chunks. At minimum, use separate cells per task, and when reasonable, separate cells for subtasks.
- Remember to in the end rerun all code from the beginning to end of the notebook to ensure each cell runs without error

## Visualizations

In the visualizations always include enough information that the plot can be understood independently. This includes:

- Labels for both axes
- A descriptive title

## Statement of use of AI

Include a brief statement describing how and which AI was used (or if no AI was used) in completing the assignment. This could be a markdown cell with a couple of sentences. As a reminder, AI use is permitted in the assignments, but it is advisable to try to complete the tasks as far as possible without and to make sure you understand the code that AI produced when using it.

## Grading

This assignment is worth 10 points. Task 0 is worth 1 point, and tasks 1-2 are worth 2 points and task 3 is worth 5 points.

Points are given only for code that runs. If the code does not run, the task (or subtask if code for a task is divided into multiple cells) will automatically receive 0 points even if the code is almost correct.

### Penalties

- **-2 points per task** where AI-generated (hallucinated) data is used instead of the actual data provided in the task or retrieved from the specified source. The assignment requires working with real data, not made-up values!
- **-3 points** if an API key is included in the submission notebook or anywhere in the GitHub repository. Store your keys in a `.env` file and add `.env` to your `.gitignore`.
- **-1 point** if the Jupyter Notebook is overall messy and not structured well (e.g. if all tasks are completed within one cell, if answers are difficult to find due to too much irrelevant printed output).
- **-1 point** if there is no statement of AI use. If you did not use AI, report that you did not use AI.

### Editing the submission after the deadline

- Editing the assignment submission during the evaluation phase is forbidden. Thus, after the solution has been released, do not make any further changes to the notebook until you have received a grade. If you accidentally leaked an API key, revoke the key immediately. Other **changes to the submission are considered cheating, and will result in 0 points for both the assignment and peer review**.

---

## Tasks

### Task 0: Setting up Ollama (1p)

a) Set up Ollama and connect to it using either openAI's API or Ollama's own API. 

b) Load the 270m parameter version of the [gemma3](https://ollama.com/library/gemma3) model and test it with any prompt.

c) Load the 4b parameter version of the [gemma3](https://ollama.com/library/gemma3) and test it with any prompt. If running the 4b version is too slow, you can use the 1b version instead.

In [4]:
import requests

#B 270m parameter test
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:270m",
        "prompt": "What is politics?",
        "stream": False
    }
)

print("270:", response.json()["response"])
#C 4b parameter test
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:1b",
        "prompt": "What is the best football club in Ukraine?",
        "stream": False
    }
)

print("1b:", response.json()["response"])

270: Politics is the political system and its processes, activities, and institutions. It's about the way governments, political parties, and other entities make decisions about how they govern and how citizens interact with them.

Okay, this is a surprisingly complex question and a really interesting one! Determining the "best" football club in Ukraine is tough because it depends on how you measure it – historical success, current performance, financial strength, fan support, and even a combination of factors.

However, based on a combination of these criteria, **Shakhtar Donetsk** is widely considered the best football club in Ukraine right now. Here's a breakdown of why:

**1. Historical Success & Title Wins:**

* **Ukrainian Super Cup:** Shakhtar has won the Ukrainian Super Cup a record 8 times, a significant achievement.
* **Ukrainian Premier League:** They've won the Ukrainian Premier League 6 times, a substantial achievement in Ukrainian football.
* **UEFA European Football Cup:

### Task 1: Text classification with Ollama (2p)

The `data/emails.csv` file contains 12 email headlines, with 4 spam emails, 4 legitimate work emails and 4 vague emails that are hard to classify based on the title alone. Use this dataset for all subtasks in this task.

a) Make a function for classifying emails (based on the headlines) as spam, work or unknown. The function should return only the classification and nothing else. (0.5p)

b) Use the smaller gemma3 (270m) to classify the emails using the function created in part a. (0.5p)

c) Use larger gemma3 (4b) to classify the emails using the function created in part a). In separate markdown cell, write a brief comment comparing the results of parts b) and c). (0.5p)

d) Write a script that repeats b) and c) 3 times, storing the results for both models separately. For both models, put the results as columns into a new DataFrame that also contains the headlines so that it is easy to compare how the output varied across runs for both models. Comment if there were differences and explain why this happened. (0.5p)

In [28]:
import pandas as pd
import requests
# A Creating a function
df = pd.read_csv("/Users/romanklymenko/Downloads/emails.csv", sep=";") #Import of dataset
print(df.head()) #See headings 
def classify_email(headline, model): #Defining function

    prompt = f""" 
You are an email classifier.

Classify the email headline into exactly one of these categories:
spam, work, unknown

Return ONLY one word. No explanation.

Headline: {headline}
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()

    if "response" in data:
        result = data["response"].strip().lower() #Making better readability 
    else:
        result = "unknown"

    return result

#B gemma3 (270m)
df["result_270m"] = df["headline"].apply(
    lambda x: classify_email(x, "gemma3:270m")
)  # Added missing closing parenthesis

#C gemma3:4b
df["result_4b"] = df["headline"].apply(
    lambda x: classify_email(x, "gemma3:4b")
)  
#D
for i in range(3):
    df[f"270m_run_{i}"] = df["headline"].apply(
        lambda x: classify_email(x, "gemma3:270m")
    )

    df[f"4b_run_{i}"] = df["headline"].apply(
        lambda x: classify_email(x, "gemma3:4b")
    )
    from IPython.display import display
    display(df[["headline", "result_270m", "result_1b"]])

                                            headline
0  URGENT: Your account will be suspended within ...
1  Congratulations! You have won a 1000€ gift car...
2  Hot singles in your area are waiting to meet y...
3  Re: Inheritance transfer of 4.5M USD pending y...
4       Meeting agenda for Thursday's project review


,headline,result_270m,result_1b
0,URGENT: Your account will be suspended within ...,suspended,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,work
4,Meeting agenda for Thursday's project review,spam,work
5,"Q3 budget report attached, please review by Fr...",spam,work
6,Reminder: Annual performance review scheduled ...,spam,work
7,"Updated draft of the manuscript, comments welcome",spam,work
8,Quick question about last week,spam,unknown
9,Following up,spam,work


,headline,result_270m,result_1b
0,URGENT: Your account will be suspended within ...,suspended,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,work
4,Meeting agenda for Thursday's project review,spam,work
5,"Q3 budget report attached, please review by Fr...",spam,work
6,Reminder: Annual performance review scheduled ...,spam,work
7,"Updated draft of the manuscript, comments welcome",spam,work
8,Quick question about last week,spam,unknown
9,Following up,spam,work


,headline,result_270m,result_1b
0,URGENT: Your account will be suspended within ...,suspended,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,work
4,Meeting agenda for Thursday's project review,spam,work
5,"Q3 budget report attached, please review by Fr...",spam,work
6,Reminder: Annual performance review scheduled ...,spam,work
7,"Updated draft of the manuscript, comments welcome",spam,work
8,Quick question about last week,spam,unknown
9,Following up,spam,work


Comparing the results of 270m and 1b gemma 3 no difference was noticed; both models displayed the same result. Both models didn't manage to categorise "URGENT: Your account will be suspended within" and "Quick question about last week" headlines, and therefore have put it to categories which I didn't ask.

### Task 2: Sentiment analysis with Ollama (2p)

The `data/news.csv` file contains 10 fictional financial news headlines. Use it for all subtasks in this task.

a) Make a function for classifying the texts in the provided dataset based on the topic (earnings, mergers, regulation, macroeconomics) and for determining the sentiment of the news (positive, negative, neutral). The function should return the class and sentiment in JSON format. (1p)

b) Use gemma3 (4b) to classify and provide the sentiment for each row of the provided dataset, inserting them into a new DataFrame that contains both the original headlines as well as topic and sentiment. (0.5p)

c) Give the same data and prompt to a browser based LLM (e.g. ChatGPT, LeChat, Claude or Gemini) and ask it to provide the topic and sentiment, giving it the same options. Paste the results into a markdown cell. Compare the results of b) and c), which one is more accurate and why? (0.5p)

In [1]:
import pandas as pd
import requests
import json
from IPython.display import display
# Load dataset
df = pd.read_csv("/Users/romanklymenko/Downloads/news.csv", sep=";")
print(df.head())
# A) Classification function
def classify_text(headline, model):

    prompt = f"""
You are a financial news classifier.

Classify the headline into:
topic: earnings, mergers, regulation, macroeconomics
sentiment: positive, negative, neutral

Return ONLY JSON in this format:
{{"topic": "...", "sentiment": "..."}}

Headline: {headline}
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )

    data = response.json()

    text = data["response"]

    try:
        result = json.loads(text)
    except:
        result = {"topic": "unknown", "sentiment": "neutral"}

    return result

# B) Run model (Gemma 4B)
results = df["headline"].apply(
    lambda x: classify_text(x, "gemma3:4b")
)

# Split JSON into columns
df["topic"] = results.apply(lambda x: x["topic"])
df["sentiment"] = results.apply(lambda x: x["sentiment"])
# Show result
display(df)
#C Copying headlines for browser prompt
headlines = df["headline"].tolist()
display (headlines)

                                            headline
0  Nordion Industries beats Q1 earnings estimates...
1  Helvora Pharmaceuticals misses earnings foreca...
2  Aurelis Bank reports steady quarterly profit, ...
3  Veridyne Logistics to acquire rival Trantec in...
4  Antitrust regulators block proposed merger bet...


,headline,topic,sentiment
0,Nordion Industries beats Q1 earnings estimates...,earnings,positive
1,Helvora Pharmaceuticals misses earnings foreca...,earnings,negative
2,"Aurelis Bank reports steady quarterly profit, ...",earnings,positive
3,Veridyne Logistics to acquire rival Trantec in...,mergers,positive
4,Antitrust regulators block proposed merger bet...,regulation,negative
5,Kestrel Semiconductor confirms early-stage mer...,mergers,neutral
6,New EU AI Act compliance rules expected to rai...,regulation,negative
7,Finnish FSA grants Norvik Capital expanded lic...,regulation,positive
8,"Eurozone inflation cools to 2.1%, easing press...",macroeconomics,neutral
9,Rising interest rates weigh on Tessaro Real Es...,macroeconomics,negative


['Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%',
 'Helvora Pharmaceuticals misses earnings forecast amid weak generics demand',
 'Aurelis Bank reports steady quarterly profit, in line with analyst expectations',
 'Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal',
 'Antitrust regulators block proposed merger between Solenta and Marvex Energy',
 'Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG',
 'New EU AI Act compliance rules expected to raise costs for Lumavex by 12%',
 'Finnish FSA grants Norvik Capital expanded licence for cross-border operations',
 'Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs',
 'Rising interest rates weigh on Tessaro Real Estate as financing costs climb']

| Headline                                                                            | Topic          | Sentiment |
| ----------------------------------------------------------------------------------- | -------------- | --------- |
| Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%          | earnings       | positive  |
| Helvora Pharmaceuticals misses earnings forecast amid weak generics demand          | earnings       | negative  |
| Aurelis Bank reports steady quarterly profit, in line with analyst expectations     | earnings       | neutral   |
| Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal                | mergers        | positive  |
| Antitrust regulators block proposed merger between Solenta and Marvex Energy        | mergers        | negative  |
| Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG           | mergers        | neutral   |
| New EU AI Act compliance rules expected to raise costs for Lumavex by 12%           | regulation     | negative  |
| Finnish FSA grants Norvik Capital expanded licence for cross-border operations      | regulation     | positive  |
| Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs | macroeconomics | positive  |
| Rising interest rates weigh on Tessaro Real Estate as financing costs climb         | macroeconomics | negative  |


COMPARISON B AND C TASKS

Browser-based LLM was more accurate than the local one. Browser-based LLM provided more consistent classifications. Otherwise, the local model occasionally produced incorrect topic classifications or fallback values like "unknown" (often in gemma3:1b). The reasons for this are that the Browser LLMs are much larger and have better training data. Unlike the local LLM, which has less data and is smaller in size.


### Task 3: Supervised machine learning (5p)

For this task, use a subset of the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) dataset, by downloading and importing the `bank-additional.csv` from the UCI repository. You can find a description of the dataset behind the link.

The goal is to predict whether a prospective customer will subscribe to a term deposit (variable y).

a) Import the dataset and conduct exploratory data analysis on it. (1p)

b) Preprocess the data using the appropriate methods as described in the course materials. (1p)

c) Determine whether this is a classification or regression task. Choose three different machine learning algorithms from scikit-learn and explain briefly why you chose them. For each of the selected algorithsm, train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance. (1p)

d) Compare using train, validation and test set split versus using cross-validation. Which one performs better? (1p)

e) Report and evaluate the performance of the models using several of the metrics provided in the course, and explain which model is the best for the task and why. (1p)


In [3]:
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# A) Loading dataset 
df = pd.read_csv("/Users/romanklymenko/Downloads/bank-additional.csv", sep=";")
#EDA
display(df.head())
display(df.info())
display(df.describe())
display(df["y"].value_counts())

# B) Preprocessing

df["y"] = df["y"].map({"yes": 1, "no": 0})

X = df.drop("y", axis=1)
y = df["y"]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# C) MODELS

# 1)Logistic Regression
log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_model.fit(X_train, y_train)
pred_log = log_model.predict(X_test)
acc_log = accuracy_score(y_test, pred_log)

# 2)Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, pred_rf)

# 3)Gradient Boosting
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)
acc_gb = accuracy_score(y_test, pred_gb)

print("\nMODEL ACCURACY")
print("LogReg:", acc_log)
print("RF    :", acc_rf)
print("GB    :", acc_gb)

# D) TRAIN and CROSS-VALIDATION


# Train/Validation/Test split
X_train2, X_temp, y_train2, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test2, y_val, y_test2 = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

rf2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

rf2.fit(X_train2, y_train2)

val_score = rf2.score(X_val, y_val)
test_score = rf2.score(X_test2, y_test2)

# Cross-validation
cv_scores = cross_val_score(rf2, X, y, cv=5)

print("\nD) COMPARISON")
print("Validation score:", val_score)
print("Test score:", test_score)
print("CV scores:", cv_scores)
print("CV mean:", cv_scores.mean())
# E) FINAL EVALUATION
pred = rf.predict(X_test)

print("\nMETRICS (Random Forest)")
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

display(pd.DataFrame(
    classification_report(y_test, pred, output_dict=True)
).transpose())

cm = confusion_matrix(y_test, pred)
display(pd.DataFrame(
    cm,
    columns=["Pred 0", "Pred 1"],
    index=["Actual 0", "Actual 1"]
))


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   object 
 2   marital         4119 non-null   object 
 3   education       4119 non-null   object 
 4   default         4119 non-null   object 
 5   housing         4119 non-null   object 
 6   loan            4119 non-null   object 
 7   contact         4119 non-null   object 
 8   month           4119 non-null   object 
 9   day_of_week     4119 non-null   object 
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   object 
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   f

None

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000,4119.000000
mean,40.113620,256.788055,2.537266,960.422190,0.190337,0.084972,93.579704,-40.499102,3.621356,5166.481695
std,10.313362,254.703736,2.568159,191.922786,0.541788,1.563114,0.579349,4.594578,1.733591,73.667904
min,18.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.635000,4963.600000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.334000,5099.100000
50%,38.000000,181.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,317.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,88.000000,3643.000000,35.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


y
no     3668
yes     451
Name: count, dtype: int64

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



MODEL ACCURACY
LogReg: 0.8495145631067961
RF    : 0.9004854368932039
GB    : 0.9065533980582524

D) COMPARISON
Validation score: 0.8948220064724919
Test score: 0.9077669902912622
CV scores: [0.89199029 0.90048544 0.90776699 0.89320388 0.90157959]
CV mean: 0.8990052377638051

METRICS (Random Forest)
Precision: 0.5480769230769231
Recall: 0.6195652173913043
F1: 0.5816326530612245


,precision,recall,f1-score,support
0,0.951389,0.935792,0.943526,732.000000
1,0.548077,0.619565,0.581633,92.000000
accuracy,0.900485,0.900485,0.900485,0.900485
macro avg,0.749733,0.777679,0.762579,824.000000
weighted avg,0.906359,0.900485,0.903121,824.000000


,Pred 0,Pred 1
Actual 0,685,47
Actual 1,35,57


### EXPLANATION PART
C) 
1) Linear regression model - used as a baseline for binary classification
2) Random Forest - used due to the ability to solve complex feature interactions
3) Gradient Boosting - used due to strong predictive performance
   
D) Cross-validation performs better because it provides a more stable and reliable estimate of model performance compared to a single train/validation/test split.

E)The models were evaluated using accuracy, precision, recall, F1-score, and confusion matrix. Gradient Boosting achieved the highest overall performance and is therefore selected as the best model for this task. However, due to class imbalance, metrics such as recall and F1-score are more important than accuracy alone when interpreting the results.

### USAGE OF AI:
AI was used with two primary tasks. First,  to choose the appropriate Models for training to understand which ones are most appropriate for the task and why. 
Second, AI was used to write some lines of code. Mainly, it concerned the TRAIN and CROSS-VALIDATION part.